In [2]:
from pathlib import Path
import pandas as pd

raw_path = Path("../data/raw")
processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

# Load
df_orders = pd.read_csv(raw_path / "orders.csv")
df_orders_products_train = pd.read_csv(raw_path / "order_products_train.csv")
df_products = pd.read_csv(raw_path / "products.csv")
df_departments = pd.read_csv(processed_path / "departments_t.csv")
df_aisles = pd.read_csv(raw_path / "aisles.csv")

# orders_train
df_orders_train = (
    df_orders.loc[df_orders["eval_set"] == "train", [
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
    ]]
    .copy()
)
df_orders_train.to_csv(processed_path / "orders_train.csv", index=False)
del df_orders

# products_departments
df_products_departments = df_products.merge(
    df_departments,
    on="department_id",
    how="left"
)
df_products_departments.to_csv(
    processed_path / "products_departments.csv",
    index=False
)
del df_departments

# products_enriched
df_products_enriched = df_products_departments.merge(
    df_aisles,
    on="aisle_id",
    how="left"
)
df_products_enriched.to_csv(
    processed_path / "products_enriched.csv",
    index=False
)
del df_products_departments, df_aisles

# train_transactions
df_train_transactions = df_orders_train.merge(
    df_orders_products_train,
    on="order_id",
    how="left"
)
df_train_transactions.to_csv(
    processed_path / "train_transactions.csv",
    index=False
)
del df_orders_products_train

# final analytical table
df_train_analytic = df_train_transactions.merge(
    df_products_enriched,
    on="product_id",
    how="left"
)
df_train_analytic.to_csv(
    processed_path / "train_analytic.csv",
    index=False
)

print("Final table shape:", df_train_analytic.shape)
print(df_train_analytic.head())

Final table shape: (1384706, 15)
   order_id  user_id  order_number  order_dow  order_hour_of_day  \
0   1187899        1            11          4                  8   
1   1187899        1            11          4                  8   
2   1187899        1            11          4                  8   
3   1187899        1            11          4                  8   
4   1187899        1            11          4                  8   

   days_since_prior_order  product_id  add_to_cart_order  reordered  \
0                    14.0         196                  1          1   
1                    14.0       25133                  2          1   
2                    14.0       38928                  3          1   
3                    14.0       26405                  4          1   
4                    14.0       39657                  5          1   

                       product_name  aisle_id  department_id  prices  \
0                              Soda      77.0            7.